# Promoter DP-SGD ε = 8, 4, 2

**Curated extraction from the original 124-cell notebook.** Outputs removed. This notebook has **not been executed end-to-end**; original cells are preserved where noted. Training is costly and changes checkpoints. Original DP-SGD training uses manual per-example gradients; review privacy assumptions and ensure per-run fresh model state.

## Environment and data
Run in **Google Colab with GPU**. Restart runtime after installing if incompatible libraries are already imported. Check paths before training; original source saves to Google Drive.

In [ ]:
%pip install -q "transformers==4.49.0" "tokenizers==0.21.4" "huggingface-hub==0.36.2" "datasets==3.6.0" "opacus==1.6.0" "pandas==2.2.3" "accelerate" "scikit-learn" "matplotlib" "tqdm" "jedi>=0.16"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/DNABERT_DP_Project/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
import random,numpy as np,torch
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
MODEL_ID = 'zhihan1996/DNABERT-2-117M'
class DNABERTBiTFiTClassifier(nn.Module):
    def __init__(self, encoder, num_labels=2):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters(): p.requires_grad = False
        for name,p in self.encoder.named_parameters():
            if name.endswith('.bias'): p.requires_grad = True
        self.classifier = nn.Linear(self.encoder.config.hidden_size,num_labels)
    def forward(self,input_ids,attention_mask=None):
        outputs=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        hidden=outputs[0]
        if attention_mask is not None:
            mask=attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
        else: pooled=hidden.mean(1)
        return self.classifier(pooled)
def fresh_encoder():
    encoder=AutoModel.from_pretrained(MODEL_ID,trust_remote_code=True,add_pooling_layer=False)
    for module in encoder.modules():
        if module.__class__.__name__=='BertUnpadSelfAttention':
            module.p_dropout=0.1
            if hasattr(module,'dropout'): module.dropout.p=0.1
    return encoder

**Original source cell 68** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 50 - RESTORE ORIGINAL PROMOTER DATASET
# Recreate the original 42,620 / 4,736 train/validation split
# ============================================================

import os
import numpy as np
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

MODEL_ID = "zhihan1996/DNABERT-2-117M"
SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 8

print("=" * 68)
print("CELL 50 - RESTORE PROMOTER DATASET")
print("=" * 68)

# ------------------------------------------------------------
# 1. Load original GUE promoter dataset
# ------------------------------------------------------------

print("\n1. LOADING GUE PROMOTER DATASET")
print("-" * 68)

promoter = load_dataset(
    "leannmlindsey/GUE",
    "prom_core_all"
)

print(promoter)

original_train = promoter["train"]
original_test = promoter["test"]

print("\nOriginal training examples:", len(original_train))
print("Original test examples:", len(original_test))

assert len(original_train) == 47356
assert len(original_test) == 5920

# ------------------------------------------------------------
# 2. Recreate stratified train/validation split
# ------------------------------------------------------------

print("\n2. RECREATING TRAIN/VALIDATION SPLIT")
print("-" * 68)

indices = np.arange(len(original_train))
labels = np.asarray(original_train["label"])

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

promoter_train = original_train.select(train_idx.tolist())
promoter_val = original_train.select(val_idx.tolist())
promoter_test = original_test

print("Training:", len(promoter_train))
print("Validation:", len(promoter_val))
print("Test:", len(promoter_test))

assert len(promoter_train) == 42620
assert len(promoter_val) == 4736
assert len(promoter_test) == 5920

# ------------------------------------------------------------
# 3. Load tokenizer
# ------------------------------------------------------------

print("\n3. TOKENIZER")
print("-" * 68)

promoter_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True
)

print("Tokenizer:", promoter_tokenizer.__class__.__name__)

# ------------------------------------------------------------
# 4. Tokenize all three splits
# ------------------------------------------------------------

def tokenize_promoter(batch):

    encoded = promoter_tokenizer(
        batch["sequence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

    encoded["labels"] = batch["label"]

    return encoded


promoter_train_dataset = promoter_train.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_train.column_names,
    desc="Tokenizing promoter training"
)

promoter_val_dataset = promoter_val.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_val.column_names,
    desc="Tokenizing promoter validation"
)

promoter_test_dataset = promoter_test.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_test.column_names,
    desc="Tokenizing promoter test"
)

# ------------------------------------------------------------
# 5. Build evaluation loaders
# ------------------------------------------------------------

promoter_collator = DataCollatorWithPadding(
    tokenizer=promoter_tokenizer,
    padding=True,
    return_tensors="pt"
)

# IMPORTANT:
# This ordinary DataLoader is for inspection/evaluation.
# It is NOT the Poisson sampler for private training.

promoter_val_loader = DataLoader(
    promoter_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=promoter_collator
)

promoter_test_loader = DataLoader(
    promoter_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=promoter_collator
)

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n4. RESTORED DATASET VERIFICATION")
print("-" * 68)

print("Training examples:", len(promoter_train_dataset))
print("Validation examples:", len(promoter_val_dataset))
print("Test examples:", len(promoter_test_dataset))

print(
    "Training label distribution:",
    np.bincount(
        np.asarray(promoter_train_dataset["labels"])
    ).tolist()
)

print(
    "Validation label distribution:",
    np.bincount(
        np.asarray(promoter_val_dataset["labels"])
    ).tolist()
)

sample = promoter_train_dataset[0]

print("Tokenized fields:", list(sample.keys()))
print("First sequence token count:", len(sample["input_ids"]))

assert len(promoter_train_dataset) == 42620
assert len(promoter_val_dataset) == 4736
assert len(promoter_test_dataset) == 5920

print("\n" + "=" * 68)
print("CELL 50 COMPLETE - PROMOTER DATASET RESTORED")
print("=" * 68)

In [ ]:
# Compatibility aliases used by the original DP-SGD training cells.
train_dataset = promoter_train_dataset
val_dataset = promoter_val_dataset
test_dataset = promoter_test_dataset
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True,collate_fn=promoter_collator)
eval_loader = promoter_val_loader
test_loader = promoter_test_loader
classifier_model = DNABERTBiTFiTClassifier(fresh_encoder(),num_labels=2).to(device)
print('Training:',len(train_dataset),'Validation:',len(val_dataset),'Test:',len(test_dataset))

**Static audit correction:** The compatibility DataLoader now uses `promoter_collator` defined by the dataset setup cell, replacing an undefined `collator` name. This notebook has not been run end-to-end.


## Original experiment cells
**CAUTION:** The following cells were extracted from different Colab sessions. Review paths, variable dependencies, privacy assumptions, and checkpoint names before running. Run one training budget at a time.

**Original source cell 26** — retained with its original logic.


In [ ]:
# Test Explicit Per-Example Gradients

import gc
import torch
import torch.nn.functional as F
from transformers import AutoModel

# Clean up expanded-weights test models
for name in ["ew_model", "ew_classifier", "ew_encoder"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Fresh pretrained DNABERT-2 encoder
manual_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

# Configure standard PyTorch attention
for module in manual_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        module.dropout.p = 0.1

# Build BiTFiT classifier
manual_model = DNABERTBiTFiTClassifier(
    manual_encoder,
    num_labels=2
).to(device)

manual_model.train()

# Get two examples
manual_batch = next(iter(train_loader))
manual_batch = {
    key: value[:2].to(device)
    for key, value in manual_batch.items()
}

trainable = [
    (name, param)
    for name, param in manual_model.named_parameters()
    if param.requires_grad
]

per_example_grads = {
    name: [] for name, _ in trainable
}

print("Testing explicit per-example gradients")
print("-" * 50)

try:
    for i in range(2):

        manual_model.zero_grad(set_to_none=True)

        logits = manual_model(
            input_ids=manual_batch["input_ids"][i:i+1],
            attention_mask=manual_batch["attention_mask"][i:i+1]
        )

        loss = F.cross_entropy(
            logits,
            manual_batch["labels"][i:i+1].long()
        )

        loss.backward()

        for name, param in trainable:
            if param.grad is None:
                raise RuntimeError(
                    f"Missing gradient for {name}"
                )

            per_example_grads[name].append(
                param.grad.detach().clone()
            )

        print(
            f"Example {i+1}: "
            f"Loss = {loss.item():.4f}, "
            f"Gradients calculated"
        )

    # Stack gradients along example dimension
    for name in per_example_grads:
        per_example_grads[name] = torch.stack(
            per_example_grads[name],
            dim=0
        )

    # Validate shapes
    correct = 0
    incorrect = []

    for name, param in trainable:
        shape = per_example_grads[name].shape

        if shape[0] == 2 and shape[1:] == param.shape:
            correct += 1
        else:
            incorrect.append((name, shape))

    print("\nExplicit Gradient Validation")
    print("-" * 50)
    print("Trainable tensors:", len(trainable))
    print("Correct gradient tensors:", correct)
    print("Incorrect gradient tensors:", len(incorrect))

    if not incorrect:
        print("\nPASS: Explicit per-example gradients calculated.")
    else:
        print("\nFAIL: Some gradient shapes are incorrect.")
        for name, shape in incorrect:
            print(name, shape)

except Exception as e:
    print("\nExplicit gradient test failed.")
    print("Error type:", type(e).__name__)
    print("Error:", str(e)[:1500])

**Original source cell 27** — retained with its original logic.


In [ ]:
# Verify Explicit Gradient Correctness

import torch
import torch.nn.functional as F

# Disable dropout so both methods are comparable
manual_model.eval()

trainable = [
    (name, param)
    for name, param in manual_model.named_parameters()
    if param.requires_grad
]

# Calculate ordinary batch gradients
manual_model.zero_grad(set_to_none=True)

logits = manual_model(
    input_ids=manual_batch["input_ids"],
    attention_mask=manual_batch["attention_mask"]
)

loss = F.cross_entropy(
    logits,
    manual_batch["labels"].long()
)

loss.backward()

batch_grads = {
    name: param.grad.detach().clone()
    for name, param in trainable
}

# Calculate individual gradients and average them
individual_grads = {
    name: [] for name, _ in trainable
}

for i in range(2):

    manual_model.zero_grad(set_to_none=True)

    logits = manual_model(
        input_ids=manual_batch["input_ids"][i:i+1],
        attention_mask=manual_batch["attention_mask"][i:i+1]
    )

    loss = F.cross_entropy(
        logits,
        manual_batch["labels"][i:i+1].long()
    )

    loss.backward()

    for name, param in trainable:
        individual_grads[name].append(
            param.grad.detach().clone()
        )

# Compare the gradients
max_difference = 0.0
failures = []

for name, param in trainable:

    average_gradient = torch.stack(
        individual_grads[name]
    ).mean(dim=0)

    batch_gradient = batch_grads[name]

    difference = (
        average_gradient - batch_gradient
    ).abs().max().item()

    max_difference = max(max_difference, difference)

    if not torch.allclose(
        average_gradient,
        batch_gradient,
        rtol=1e-3,
        atol=1e-4
    ):
        failures.append((name, difference))

print("Gradient Correctness Validation")
print("-" * 50)
print("Parameters tested:", len(trainable))
print("Maximum absolute difference:", max_difference)
print("Parameters failing comparison:", len(failures))

if len(failures) == 0:
    print("\nPASS: Explicit gradients match batch gradients.")
else:
    print("\nFAIL: Gradient differences detected.")
    for name, difference in failures[:10]:
        print(name, difference)

manual_model.zero_grad(set_to_none=True)

**Original source cell 28** — retained with its original logic.


In [ ]:
# Configure DP-SGD Privacy Accounting

import math
from opacus.accountants.utils import get_noise_multiplier
from opacus.accountants import RDPAccountant

# Privacy settings
DELTA = 1e-5
TARGET_EPSILONS = [2.0, 4.0, 8.0]

DP_EPOCHS = 3
DP_BATCH_SIZE = 8
MAX_GRAD_NORM = 1.0

# Dataset size
N_TRAIN = len(train_dataset)

# Poisson sampling probability
SAMPLE_RATE = DP_BATCH_SIZE / N_TRAIN

# Number of privacy-accounted steps
STEPS_PER_EPOCH = math.ceil(N_TRAIN / DP_BATCH_SIZE)
TOTAL_STEPS = DP_EPOCHS * STEPS_PER_EPOCH

print("DP-SGD Privacy Configuration")
print("-" * 50)
print("Training examples:", N_TRAIN)
print("Target batch size:", DP_BATCH_SIZE)
print("Sample rate:", round(SAMPLE_RATE, 6))
print("Training epochs:", DP_EPOCHS)
print("Total steps:", TOTAL_STEPS)
print("Delta:", DELTA)
print("Maximum gradient norm:", MAX_GRAD_NORM)

noise_multipliers = {}

for epsilon in TARGET_EPSILONS:

    sigma = get_noise_multiplier(
        target_epsilon=epsilon,
        target_delta=DELTA,
        sample_rate=SAMPLE_RATE,
        steps=TOTAL_STEPS,
        accountant="rdp"
    )

    noise_multipliers[epsilon] = sigma

    accountant = RDPAccountant()

    for _ in range(TOTAL_STEPS):
        accountant.step(
            noise_multiplier=sigma,
            sample_rate=SAMPLE_RATE
        )

    calculated_epsilon = accountant.get_epsilon(
        delta=DELTA
    )

    print(
        f"\nTarget epsilon: {epsilon:.1f}"
        f"\nNoise multiplier: {sigma:.4f}"
        f"\nCalculated epsilon: {calculated_epsilon:.4f}"
    )

**Original source cell 30** — retained with its original logic.


In [ ]:
# Full DP-SGD Training (Target epsilon = 8)

import gc
import time
import math
import copy
import torch
import torch.nn.functional as F

from transformers import AutoModel
from opacus.accountants import RDPAccountant
from torch.utils.data import DataLoader

TARGET_EPSILON = 8.0
sigma = noise_multipliers[TARGET_EPSILON]

# Remove the diagnostic model
for name in ["dp_model", "dp_encoder", "optimizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Fresh initialization
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

dp_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

for module in dp_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        module.dropout.p = 0.1

dp_model = DNABERTBiTFiTClassifier(
    dp_encoder,
    num_labels=2
).to(device)

# Match the diagnostic's dropout policy
dp_model.eval()

trainable = [
    (name, p)
    for name, p in dp_model.named_parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    [p for _, p in trainable],
    lr=2e-4
)

accountant = RDPAccountant()

sampling_generator = torch.Generator(device="cpu")
sampling_generator.manual_seed(12345)

# Use the same number of steps as Cell 20
steps_per_epoch = STEPS_PER_EPOCH

dp_history = []
best_accuracy = -1.0
best_state = None
best_step = None

start_time = time.time()

print("Full DP-SGD Training")
print("-" * 55)
print("Target epsilon:", TARGET_EPSILON)
print("Noise multiplier:", round(sigma, 4))
print("Total steps:", TOTAL_STEPS)
print("Trainable parameters:", sum(p.numel() for _, p in trainable))

for epoch in range(DP_EPOCHS):

    epoch_losses = []
    epoch_clipped = 0
    epoch_samples = 0

    for step in range(steps_per_epoch):

        # Independent Poisson selection
        selected = torch.nonzero(
            torch.rand(
                N_TRAIN,
                generator=sampling_generator
            ) < SAMPLE_RATE,
            as_tuple=True
        )[0].tolist()

        clipped_sums = [
            torch.zeros_like(p)
            for _, p in trainable
        ]

        for idx in selected:

            example = train_dataset[idx]

            input_ids = torch.tensor(
                [example["input_ids"]],
                dtype=torch.long,
                device=device
            )

            attention_mask = torch.tensor(
                [example["attention_mask"]],
                dtype=torch.long,
                device=device
            )

            labels = torch.tensor(
                [example["labels"]],
                dtype=torch.long,
                device=device
            )

            dp_model.zero_grad(set_to_none=True)

            logits = dp_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            grad_norm = torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(p.grad.detach())
                    for _, p in trainable
                ])
            )

            factor = min(
                1.0,
                MAX_GRAD_NORM / (grad_norm.item() + 1e-12)
            )

            if factor < 1.0:
                epoch_clipped += 1

            for j, (_, p) in enumerate(trainable):
                clipped_sums[j].add_(
                    p.grad.detach(),
                    alpha=factor
                )

            epoch_losses.append(loss.item())
            epoch_samples += 1

        optimizer.zero_grad(set_to_none=True)

        # Gaussian mechanism; also runs for empty batches
        with torch.no_grad():
            for j, (_, p) in enumerate(trainable):
                noise = torch.randn_like(p) * (
                    sigma * MAX_GRAD_NORM
                )

                p.grad = (
                    clipped_sums[j] + noise
                ) / DP_BATCH_SIZE

        optimizer.step()

        accountant.step(
            noise_multiplier=sigma,
            sample_rate=SAMPLE_RATE
        )

    # Validation after each epoch
    dp_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_val in eval_loader:

            ids = batch_val["input_ids"].to(device)
            mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device).long()

            logits = dp_model(
                input_ids=ids,
                attention_mask=mask
            )

            val_loss += (
                F.cross_entropy(
                    logits, labels, reduction="sum"
                ).item()
            )

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= val_total

    current_epsilon = accountant.get_epsilon(delta=DELTA)
    elapsed = time.time() - start_time

    dp_history.append({
        "epoch": epoch + 1,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
        "epsilon": current_epsilon,
        "samples_processed": epoch_samples,
        "clipped": epoch_clipped,
        "elapsed_seconds": elapsed
    })

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_step = (epoch + 1) * steps_per_epoch

        # Save checkpoint to CPU
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in dp_model.state_dict().items()
        }

    print(
        f"Epoch {epoch+1}/{DP_EPOCHS} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Epsilon: {current_epsilon:.4f} | "
        f"Time: {elapsed/60:.1f} min",
        flush=True
    )

print("\nDP-SGD Training Complete")
print("Best validation accuracy:", round(best_accuracy, 4))
print("Best checkpoint step:", best_step)
print("Final training epsilon:", accountant.get_epsilon(DELTA))
print("Total runtime (minutes):", round((time.time()-start_time)/60, 1))

**Original source cell 33** — retained with its original logic.


In [ ]:
# Full DP-SGD Training (epsilon = 4)

import os
import gc
import json
import time
import torch
import torch.nn.functional as F
import pandas as pd

from transformers import AutoModel
from opacus.accountants import RDPAccountant

TARGET_EPSILON = 4.0
sigma = noise_multipliers[TARGET_EPSILON]

RESULTS_DIR = "/content/dp_dnabert_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Free previous experiment's GPU memory
for name in ["dp_model", "dp_encoder", "optimizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Initialize from the same pretrained checkpoint
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

dp_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

for module in dp_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        module.dropout.p = 0.1

dp_model = DNABERTBiTFiTClassifier(
    dp_encoder,
    num_labels=2
).to(device)

# Match epsilon = 8 training configuration
dp_model.eval()

trainable = [
    (name, p)
    for name, p in dp_model.named_parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    [p for _, p in trainable],
    lr=2e-4
)

accountant = RDPAccountant()

sampling_generator = torch.Generator(device="cpu")
sampling_generator.manual_seed(12345)

dp_history = []
best_accuracy = -1.0
best_step = None

start_time = time.time()

print("Full DP-SGD Training: epsilon = 4")
print("-" * 55)
print("Noise multiplier:", round(sigma, 4))
print("Total steps:", TOTAL_STEPS)

for epoch in range(DP_EPOCHS):

    epoch_losses = []
    epoch_clipped = 0
    epoch_samples = 0

    for step in range(STEPS_PER_EPOCH):

        # Independent Poisson sampling
        selected = torch.nonzero(
            torch.rand(
                N_TRAIN,
                generator=sampling_generator
            ) < SAMPLE_RATE,
            as_tuple=True
        )[0].tolist()

        clipped_sums = [
            torch.zeros_like(p)
            for _, p in trainable
        ]

        for idx in selected:

            example = train_dataset[idx]

            input_ids = torch.tensor(
                [example["input_ids"]],
                dtype=torch.long,
                device=device
            )

            attention_mask = torch.tensor(
                [example["attention_mask"]],
                dtype=torch.long,
                device=device
            )

            labels = torch.tensor(
                [example["labels"]],
                dtype=torch.long,
                device=device
            )

            dp_model.zero_grad(set_to_none=True)

            logits = dp_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            grad_norm = torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(p.grad.detach())
                    for _, p in trainable
                ])
            )

            factor = min(
                1.0,
                MAX_GRAD_NORM / (grad_norm.item() + 1e-12)
            )

            if factor < 1.0:
                epoch_clipped += 1

            for j, (_, p) in enumerate(trainable):
                clipped_sums[j].add_(
                    p.grad.detach(),
                    alpha=factor
                )

            epoch_losses.append(loss.item())
            epoch_samples += 1

        optimizer.zero_grad(set_to_none=True)

        # Add Gaussian noise
        with torch.no_grad():
            for j, (_, p) in enumerate(trainable):
                noise = torch.randn_like(p) * (
                    sigma * MAX_GRAD_NORM
                )

                p.grad = (
                    clipped_sums[j] + noise
                ) / DP_BATCH_SIZE

        optimizer.step()

        accountant.step(
            noise_multiplier=sigma,
            sample_rate=SAMPLE_RATE
        )

    # Validation
    dp_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_val in eval_loader:

            ids = batch_val["input_ids"].to(device)
            mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device).long()

            logits = dp_model(
                input_ids=ids,
                attention_mask=mask
            )

            val_loss += F.cross_entropy(
                logits, labels, reduction="sum"
            ).item()

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= val_total

    current_epsilon = accountant.get_epsilon(delta=DELTA)
    elapsed = time.time() - start_time

    dp_history.append({
        "epoch": epoch + 1,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
        "epsilon": current_epsilon,
        "samples_processed": epoch_samples,
        "clipped": epoch_clipped,
        "elapsed_seconds": elapsed
    })

    # Save best checkpoint directly to disk
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy
        best_step = (epoch + 1) * STEPS_PER_EPOCH

        torch.save(
            dp_model.state_dict(),
            os.path.join(
                RESULTS_DIR,
                "dnabert_dp4_best.pt"
            )
        )

    print(
        f"Epoch {epoch+1}/{DP_EPOCHS} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Epsilon: {current_epsilon:.4f} | "
        f"Time: {elapsed/60:.1f} min",
        flush=True
    )

# Save final checkpoint
torch.save(
    dp_model.state_dict(),
    os.path.join(RESULTS_DIR, "dnabert_dp4_final.pt")
)

# Preserve results
results_dp4 = {
    "target_epsilon": TARGET_EPSILON,
    "noise_multiplier": float(sigma),
    "final_epsilon": float(accountant.get_epsilon(delta=DELTA)),
    "best_validation_accuracy": float(best_accuracy),
    "best_checkpoint_step": int(best_step),
    "training_history": dp_history
}

with open(
    os.path.join(RESULTS_DIR, "results_dp4.json"), "w"
) as f:
    json.dump(results_dp4, f, indent=2)

pd.DataFrame(dp_history).to_csv(
    os.path.join(RESULTS_DIR, "history_dp4.csv"),
    index=False
)

print("\nDP-SGD epsilon = 4 Complete")
print("Best validation accuracy:", f"{best_accuracy:.2%}")
print("Final epsilon:", f"{results_dp4['final_epsilon']:.4f}")
print("Total runtime:", f"{(time.time()-start_time)/60:.1f} minutes")
print("Files saved to:", RESULTS_DIR)

**Original source cell 34** — retained with its original logic.


In [ ]:
# Full DP-SGD Training (epsilon = 2)

import os
import gc
import json
import time
import torch
import torch.nn.functional as F
import pandas as pd

from transformers import AutoModel
from opacus.accountants import RDPAccountant

TARGET_EPSILON = 2.0
sigma = noise_multipliers[TARGET_EPSILON]

RESULTS_DIR = "/content/dp_dnabert_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Free previous experiment's GPU memory
for name in ["dp_model", "dp_encoder", "optimizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Initialize from the same pretrained checkpoint
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

dp_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

for module in dp_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        module.dropout.p = 0.1

dp_model = DNABERTBiTFiTClassifier(
    dp_encoder,
    num_labels=2
).to(device)

# Match epsilon = 8 training configuration
dp_model.eval()

trainable = [
    (name, p)
    for name, p in dp_model.named_parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    [p for _, p in trainable],
    lr=2e-4
)

accountant = RDPAccountant()

sampling_generator = torch.Generator(device="cpu")
sampling_generator.manual_seed(12345)

dp_history = []
best_accuracy = -1.0
best_step = None

start_time = time.time()

print("Full DP-SGD Training: epsilon = 2")
print("-" * 55)
print("Noise multiplier:", round(sigma, 4))
print("Total steps:", TOTAL_STEPS)

for epoch in range(DP_EPOCHS):

    epoch_losses = []
    epoch_clipped = 0
    epoch_samples = 0

    for step in range(STEPS_PER_EPOCH):

        # Independent Poisson sampling
        selected = torch.nonzero(
            torch.rand(
                N_TRAIN,
                generator=sampling_generator
            ) < SAMPLE_RATE,
            as_tuple=True
        )[0].tolist()

        clipped_sums = [
            torch.zeros_like(p)
            for _, p in trainable
        ]

        for idx in selected:

            example = train_dataset[idx]

            input_ids = torch.tensor(
                [example["input_ids"]],
                dtype=torch.long,
                device=device
            )

            attention_mask = torch.tensor(
                [example["attention_mask"]],
                dtype=torch.long,
                device=device
            )

            labels = torch.tensor(
                [example["labels"]],
                dtype=torch.long,
                device=device
            )

            dp_model.zero_grad(set_to_none=True)

            logits = dp_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            grad_norm = torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(p.grad.detach())
                    for _, p in trainable
                ])
            )

            factor = min(
                1.0,
                MAX_GRAD_NORM / (grad_norm.item() + 1e-12)
            )

            if factor < 1.0:
                epoch_clipped += 1

            for j, (_, p) in enumerate(trainable):
                clipped_sums[j].add_(
                    p.grad.detach(),
                    alpha=factor
                )

            epoch_losses.append(loss.item())
            epoch_samples += 1

        optimizer.zero_grad(set_to_none=True)

        # Add Gaussian noise
        with torch.no_grad():
            for j, (_, p) in enumerate(trainable):
                noise = torch.randn_like(p) * (
                    sigma * MAX_GRAD_NORM
                )

                p.grad = (
                    clipped_sums[j] + noise
                ) / DP_BATCH_SIZE

        optimizer.step()

        accountant.step(
            noise_multiplier=sigma,
            sample_rate=SAMPLE_RATE
        )

    # Validation
    dp_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_val in eval_loader:

            ids = batch_val["input_ids"].to(device)
            mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device).long()

            logits = dp_model(
                input_ids=ids,
                attention_mask=mask
            )

            val_loss += F.cross_entropy(
                logits, labels, reduction="sum"
            ).item()

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= val_total

    current_epsilon = accountant.get_epsilon(delta=DELTA)
    elapsed = time.time() - start_time

    dp_history.append({
        "epoch": epoch + 1,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
        "epsilon": current_epsilon,
        "samples_processed": epoch_samples,
        "clipped": epoch_clipped,
        "elapsed_seconds": elapsed
    })

    # Save best checkpoint directly to disk
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy
        best_step = (epoch + 1) * STEPS_PER_EPOCH

        torch.save(
            dp_model.state_dict(),
            os.path.join(
                RESULTS_DIR,
                "dnabert_dp2_best.pt"
            )
        )

    print(
        f"Epoch {epoch+1}/{DP_EPOCHS} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Epsilon: {current_epsilon:.4f} | "
        f"Time: {elapsed/60:.1f} min",
        flush=True
    )

# Save final checkpoint
torch.save(
    dp_model.state_dict(),
    os.path.join(RESULTS_DIR, "dnabert_dp2_final.pt")
)

# Preserve results
results_dp4 = {
    "target_epsilon": TARGET_EPSILON,
    "noise_multiplier": float(sigma),
    "final_epsilon": float(accountant.get_epsilon(delta=DELTA)),
    "best_validation_accuracy": float(best_accuracy),
    "best_checkpoint_step": int(best_step),
    "training_history": dp_history
}

with open(
    os.path.join(RESULTS_DIR, "results_dp2.json"), "w"
) as f:
    json.dump(results_dp4, f, indent=2)

pd.DataFrame(dp_history).to_csv(
    os.path.join(RESULTS_DIR, "history_dp2.csv"),
    index=False
)

print("\nDP-SGD epsilon = 2 Complete")
print("Best validation accuracy:", f"{best_accuracy:.2%}")
print("Final epsilon:", f"{results_dp4['final_epsilon']:.4f}")
print("Total runtime:", f"{(time.time()-start_time)/60:.1f} minutes")
print("Files saved to:", RESULTS_DIR)

## Verification status
- Notebook JSON and code-cell Python syntax were checked (excluding Colab magics).
- **No GPU training or end-to-end runtime verification was performed.**
- Checkpoint references and privacy accounting are retained from original experiments, not independently certified.
- Avoid rerunning if a completed checkpoint exists.